## What is Scaffold Split Validation?

Scaffold split validation is a dataset splitting strategy in which molecules are divided according to their molecular scaffolds rather than randomly.

All molecules sharing the same scaffold are assigned to either the training set or the testing set, but never both.

This reduces structural overlap between the two datasets and provides a more challenging and realistic model evaluation.

## Why Do We Need Scaffold Split Validation?

Machine learning models must be evaluated on molecules they have never seen before.

A common approach is to randomly divide a dataset into training and testing sets.

However, random splitting often places structurally similar molecules in both sets.

As a result, the model appears to perform well because it has already learned very similar chemical structures.

Scaffold split validation avoids this problem by ensuring that molecules sharing the same molecular scaffold remain in the same dataset.

This provides a much more realistic evaluation of a model's ability to generalize to new chemical scaffolds.

In [1]:
# Example
import pandas as pd

# Load ESOL dataset
df = pd.read_csv("esol_raw.csv")

# View first five rows
df.head()

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1


In [2]:
df = df[["smiles", "measured log solubility in mols per litre"]]

df.head()

,smiles,measured log solubility in mols per litre
0,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,-0.77
1,Cc1occc1C(=O)Nc2ccccc2,-3.30
2,CC(C)=CCCC(C)=CC(=O),-2.06
3,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,-7.87
4,c1ccsc1,-1.33


In [3]:
from rdkit import Chem

# Convert SMILES to RDKit molecule objects
df["Mol"] = df["smiles"].apply(Chem.MolFromSmiles)

# Remove invalid molecules
df = df[df["Mol"].notnull()]

print("Number of molecules:", len(df))

Number of molecules: 1128


In [4]:
# Generate Murcko Scaffold 
from rdkit.Chem.Scaffolds import MurckoScaffold

df["Scaffold"] = df["Mol"].apply(
    lambda mol: Chem.MolToSmiles(
        MurckoScaffold.GetScaffoldForMol(mol)
    )
)

df.head()

,smiles,measured log solubility in mols per litre,Mol,Scaffold
0,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,-0.77,<rdkit.Chem.rdchem.Mol object at 0x000001693F0...,c1ccc(COC2CCCC(COC3CCCCO3)O2)cc1
1,Cc1occc1C(=O)Nc2ccccc2,-3.30,<rdkit.Chem.rdchem.Mol object at 0x000001693F4...,O=C(Nc1ccccc1)c1ccoc1
2,CC(C)=CCCC(C)=CC(=O),-2.06,<rdkit.Chem.rdchem.Mol object at 0x000001693F4...,
3,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,-7.87,<rdkit.Chem.rdchem.Mol object at 0x000001693F4...,c1ccc2c(c1)ccc1c2ccc2c3ccccc3ccc21
4,c1ccsc1,-1.33,<rdkit.Chem.rdchem.Mol object at 0x000001693F4...,c1ccsc1


In [5]:
#  Count Total number of unique scaffolds
unique_scaffolds = df["Scaffold"].nunique()

print("Total Molecules :", len(df))
print("Unique Scaffolds :", unique_scaffolds)

Total Molecules : 1128
Unique Scaffolds : 269


In [6]:
# Top 10 Most Common Scaffold

scaffold_counts = (
    df["Scaffold"]
    .value_counts()
    .reset_index()
)

scaffold_counts.columns = ["Scaffold", "Frequency"]

scaffold_counts.head(10)

,Scaffold,Frequency
0,,317
1,c1ccccc1,254
2,c1ccc(-c2ccccc2)cc1,39
3,c1ccc2ccccc2c1,22
4,O=C1CC(=O)NC(=O)N1,21
5,O=C1C=C2CCC3C4CCCC4CCC3C2CC1,17
6,c1ncncn1,16
7,c1ccncc1,14
8,c1ccc(Cc2ccccc2)cc1,12
9,C1CCCCC1,10


## Interpretation

The ESOL dataset contains 1128 molecules with 269 unique Murcko scaffolds, indicating a chemically diverse dataset.

Interestingly, 317 molecules have an empty scaffold. This occurs because the Bemis–Murcko algorithm retains only ring systems and the linkers connecting them.

Molecules without any ring system (for example, simple aliphatic compounds) have no scaffold to retain, resulting in an empty scaffold representation.

The most common non-empty scaffold is the **benzene ring (c1ccccc1)**, which appears254 moleculeses**, reflecting the widespread use of aromatic compounds in chemistry and drug discovery.

In [7]:
# Scoffold Split
# Group molecules by scaffold

scaffold_groups = df.groupby("Scaffold").groups

print("Number of Scaffold Groups:", len(scaffold_groups))

Number of Scaffold Groups: 269


In [8]:
# Display the first five scaffold groups
for scaffold, indices in list(scaffold_groups.items())[:5]:
    print("=" * 60)
    print("Scaffold:", scaffold)
    print("Number of Molecules:", len(indices))
    print("Indices:", list(indices)[:10])

Scaffold: 
Number of Molecules: 317
Indices: [2, 12, 14, 15, 16, 19, 22, 25, 29, 31]
Scaffold: C(=Cc1ccccc1)c1ccccc1
Number of Molecules: 1
Indices: [95]
Scaffold: C(=Nc1ccccc1)NC=Nc1ccccc1
Number of Molecules: 1
Indices: [382]
Scaffold: C1=C2CCC3C4CCCC4CCC3C2Cc2cnoc21
Number of Molecules: 1
Indices: [552]
Scaffold: C1=C2CCCCC2C2CCC3CCCC3C2C1
Number of Molecules: 1
Indices: [172]


In [9]:
# Split Scaffold Groups into Train and Test Split

import random

# Make results reproducible
random.seed(42)

# Get all scaffold groups
scaffold_groups = list(df.groupby("Scaffold").groups.values())

# Shuffle scaffold groups
random.shuffle(scaffold_groups)

# 80% scaffold groups for training
split_index = int(0.8 * len(scaffold_groups))

train_groups = scaffold_groups[:split_index]
test_groups = scaffold_groups[split_index:]

In [10]:
# Collect Molecule Indices
# Flatten scaffold groups into molecule indices

train_indices = [idx for group in train_groups for idx in group]
test_indices = [idx for group in test_groups for idx in group]

In [11]:
# Create Train and Test DataFrames
train_df = df.loc[train_indices].reset_index(drop=True)
test_df = df.loc[test_indices].reset_index(drop=True)

print("Training molecules :", len(train_df))
print("Testing molecules  :", len(test_df))

Training molecules : 807
Testing molecules  : 321


In [12]:
# Verify there is no Scaffold Overlap

train_scaffolds = set(train_df["Scaffold"])
test_scaffolds = set(test_df["Scaffold"])

overlap = train_scaffolds.intersection(test_scaffolds)

print("Number of overlapping scaffolds:", len(overlap))

Number of overlapping scaffolds: 0


## Interpretation

The scaffold split successfully separated the dataset into training and testing sets without any shared scaffold families.

Unlike random splitting, where structurally similar molecules may appear in both datasets, scaffold splitting ensures that the model is evaluated on chemically distinct scaffolds.

This provides a more realistic estimate of how well the model will generalize to novel chemical series encountered during drug discovery.